In [1]:
pip install numpy scipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import math, numpy as np
from scipy import stats 

In [ ]:
class BlockingSystem():
    def __init__(self, arrival_sampler, service_mean=8, num_servers=10, num_customers=10*10000, service_sampler = None):
        self.arrival_sampler = arrival_sampler
        self.service_mean = service_mean
        self.service_sampler = service_sampler if service_sampler else lambda: np.random.exponential(service_mean)
        self.num_servers = num_servers
        self.num_customers = num_customers
        self.time = 0
        self.servers = [(0, 0)] * num_servers # (start time, service time)
        self.blocked = 0

    def sample_customer(self):
        arrival_delta = self.arrival_sampler()
        service_time = self.service_sampler()

        self.time += arrival_delta

        return arrival_delta, service_time, self.time 

    def start_processing(self, customer, server):
        arrival_delta, service_time, arrived_timestamp = customer
        start_time = arrived_timestamp
        
        self.servers[server] = [start_time, service_time]

    def get_first_available_server(self):
        for i in range(len(self.servers)):
            if (self.servers[i][0] + self.servers[i][1] <= self.time):
                return i
        
        return -1

    
    def simulate(self):
        while self.num_customers > 0:
            customer = self.sample_customer()
            server = self.get_first_available_server()

            if (server == -1):
                self.blocked += 1
            else:
                self.start_processing(customer, server)

            self.num_customers -= 1

        return self.blocked

def erlang_b_formula(arrival_mean, service_mean, m):
    A = arrival_mean * service_mean
    den = 0
    for i in range(m+1):
        den += (A**i / math.factorial(i))

    return ((A**m)/math.factorial(m))/den
        

arrival_mean = 1
service_mean = 8
m = 10
num_customers = 10000

blocking_system = BlockingSystem(lambda : np.random.exponential(arrival_mean),8,10,num_customers)
print(f"Blocked from simulation: {blocking_system.simulate()/num_customers}")
print(f"Blocked from erlang b formula: {erlang_b_formula(arrival_mean, service_mean, m)}")

Blocked from simulation: 0.11640
Blocked from erlang b formula: 0.12166


In [ ]:
def confidence_interval(num_reps=10, conf=0.95, **kwargs):
    estimates = np.array([
        BlockingSystem(**kwargs).simulate() / kwargs['num_customers'] 
        for _ in range(num_reps)
    ])
    mean = estimates.mean()
    sem  = estimates.std(ddof=1) / np.sqrt(num_reps)
    t    = stats.t.ppf((1 + conf) / 2, df=num_reps - 1)
    return mean, (mean - t * sem, mean + t * sem)

mean, (lo, hi) = confidence_interval(
    num_reps=10, arrival_sampler=lambda : np.random.exponential(1), service_mean=8,
    num_servers=10, num_customers=10000,
)

print(f"MEAN: {mean}")
print(f"IC 95%: [{lo}, {hi}]")

MEAN: 0.12272
IC 95%: [0.11729, 0.12815]


In [23]:
# part 2

# a
k = 4  
erlang_sampler = lambda: np.random.gamma(shape=k, scale=1/(k*1.0))

mean, (lo, hi) = confidence_interval(
    num_reps=100, arrival_sampler=erlang_sampler, service_mean=8,
    num_servers=10, num_customers=10000,
)

print(f"MEAN: {mean}")
print(f"IC 95%: [{lo}, {hi}]")

# b
def hyperexp_sampler():
    if np.random.rand() < 0.8:
        return np.random.exponential(1/0.8333)
    else:
        return np.random.exponential(1/5.0)

mean, (lo, hi) = confidence_interval(
    num_reps=100, arrival_sampler=hyperexp_sampler, service_mean=8,
    num_servers=10, num_customers=10000,
)

print(f"MEAN: {mean}")
print(f"IC 95%: [{lo}, {hi}]")

MEAN: 0.076824
IC 95%: [0.07587081571714226, 0.07777718428285775]
MEAN: 0.137989
IC 95%: [0.13652722714206963, 0.13945077285793037]


In [24]:
def pareto_sampler(k):
    scale = 8 * (k - 1) / k
    return (np.random.pareto(k) + 1) * scale

for label, sampler in [
    ("Constant",      lambda: 8.0),
    ("Pareto k=1.05", lambda: pareto_sampler(1.05)),
    ("Pareto k=2.05", lambda: pareto_sampler(2.05)),
    ("Uniform",       lambda: np.random.uniform(0, 16)),
]:
    mean, (lo, hi) = confidence_interval(
        num_reps=100, arrival_sampler=lambda: np.random.exponential(1),
        service_mean=8, service_sampler=sampler, num_servers=10, num_customers=10000,
    )
    print(f"{label} \t MEAN: {mean}, IC 95%: [{lo}, {hi}]")

Constant 	 MEAN: 0.122256, IC 95%: [0.12140155791256119, 0.12311044208743882]
Pareto k=1.05 	 MEAN: 0.0014100000000000002, IC 95%: [0.0010976461324801135, 0.001722353867519887]
Pareto k=2.05 	 MEAN: 0.120717, IC 95%: [0.11933771350465949, 0.12209628649534052]
Uniform 	 MEAN: 0.120837, IC 95%: [0.11986128693446298, 0.12181271306553702]


In [ ]:
# part 4

# For all part 3 CIs, there is an overlap between the CI and Erlang-B,
# regardless of the service time distribution I used. 
# For part 2, varying the inter-arrival distribution changes the blocking probability: Erlang arrivals (less variable)
# have lower blocking probability than Poisson arrivals, whereas hyperexponential
# arrivals which are very variable, increase it.